Heuristically counting overcommitment/undercommitment rate from the experiments.

In [2]:
from pathlib import Path

from redel.utils import read_jsonl

# define base experiments path
EXPERIMENTS = Path("/Users/andrew/Desktop/Code/_penn/redel-experiments/experiments")


def is_overcommitted(fp, overcommitment_threshold):
    """A system is overcommitted if it has <= overcommitment_threshold nodes"""
    with open(fp) as f:
        state = json.load(f)
    return len(state["state"]) <= overcommitment_threshold


def is_undercommitted(fp, undercommitment_threshold):
    """A system is undercommitted if it has any undercommitment_threshold len chain of nodes with 0 or 1 children"""
    with open(fp) as f:
        state = json.load(f)

    nodes = {node["id"]: node for node in state["state"]}
    root = next(node for node in state["state"] if node["depth"] == 0)

    # DFS into each node, when reaching leaf mark as T/F if accumulated 1-child parents >= undercommitment_threshold
    # then every node's T/F value = any(children)
    # return root node's value
    def uc_search(node, chain):
        is_chain = len(node["children"]) <= 1
        if not node["children"]:
            return chain + 1 >= undercommitment_threshold

        is_uc = False
        for child_id in node["children"]:
            child = nodes[child_id]
            is_uc = is_uc or uc_search(child, chain + 1 if is_chain else 0)
        return is_uc

    return uc_search(root, 0)

In [3]:
import json
from dataclasses import dataclass


@dataclass
class CommitmentResult:
    oc_count: int
    uc_count: int
    samples: int
    oc_ids: list[str]
    uc_ids: list[str]


def count_system(fp, overcommitment_threshold=2, undercommitment_threshold=3):
    if not fp.exists():
        return
    # get all state paths in system
    state_paths = []
    for result in read_jsonl(fp / "results.jsonl"):
        state_paths.append(fp / Path(result["log_dir"]).stem / "state.json")

    n = len(state_paths)
    if not n:
        print(f"========== {fp} ==========")
        print("no results found :(")
        return
    oc_count = 0
    uc_count = 0
    oc_ids = []
    uc_ids = []
    for state_path in state_paths:
        if is_overcommitted(state_path, overcommitment_threshold):
            oc_count += 1
            oc_ids.append(state_path.parent.name)
        if is_undercommitted(state_path, undercommitment_threshold):
            uc_count += 1
            uc_ids.append(state_path.parent.name)

    print(f"========== {fp} ==========")
    print(f"Overcommitment rate: {oc_count / n} ({oc_count} / {n})")
    print(f"Undercommitment rate: {uc_count / n} ({uc_count} / {n})")
    return CommitmentResult(oc_count=oc_count, uc_count=uc_count, samples=n, oc_ids=oc_ids, uc_ids=uc_ids)

In [4]:
fo = count_system(EXPERIMENTS / Path("fanoutqa/dev/trial2/full"))
tp = count_system(EXPERIMENTS / Path("travelplanner/validation/full"))
wa = count_system(EXPERIMENTS / Path("webarena/test/full"))

========== /Users/andrew/Desktop/Code/_penn/redel-experiments/experiments/fanoutqa/dev/trial2/full ==========
Overcommitment rate: 0.22653721682847897 (70 / 309)
Undercommitment rate: 0.11326860841423948 (35 / 309)


In [4]:
oc_total = fo.oc_count + tp.oc_count + wa.oc_count
uc_total = fo.uc_count + tp.uc_count + wa.uc_count
n_total = fo.samples + tp.samples + wa.samples

print(f"Total overcommitment rate: {oc_total / n_total} ({oc_total} / {n_total})")
print(f"Total undercommitment rate: {uc_total / n_total} ({uc_total} / {n_total})")

AttributeError: 'NoneType' object has no attribute 'oc_count'

In [5]:
import pandas as pd

df = pd.DataFrame(columns=["model", "benchmark", "system", "n", "oc_count", "oc_rate", "uc_count", "uc_rate"])
for model in [
    "cohere-hf",
    "glm",
    "gpt-oss",
    "openai",
    "qwen3",
]:
    for benchmark in ["fanoutqa", "travelplanner", "webarena"]:
        for system in [
            "full",
            "root-fc",
            "baseline",
            "small-leaf",
            "small-all",
            "small-baseline",
            "short-context",
            "short-baseline",
        ]:
            result = count_system(EXPERIMENTS / benchmark / model / system)
            if result is None:
                continue
            df.loc[len(df)] = [
                model,
                benchmark,
                system,
                result.samples,
                result.oc_count,
                result.oc_count / result.samples,
                result.uc_count,
                result.uc_count / result.samples,
            ]

========== /Users/andrew/Desktop/Code/_penn/redel-experiments/experiments/fanoutqa/cohere-hf/full ==========
Overcommitment rate: 1.0 (310 / 310)
Undercommitment rate: 0.0 (0 / 310)
========== /Users/andrew/Desktop/Code/_penn/redel-experiments/experiments/fanoutqa/cohere-hf/root-fc ==========
Overcommitment rate: 0.9741935483870968 (302 / 310)
Undercommitment rate: 0.0 (0 / 310)
========== /Users/andrew/Desktop/Code/_penn/redel-experiments/experiments/fanoutqa/cohere-hf/baseline ==========
Overcommitment rate: 1.0 (310 / 310)
Undercommitment rate: 0.0 (0 / 310)
========== /Users/andrew/Desktop/Code/_penn/redel-experiments/experiments/fanoutqa/cohere-hf/small-all ==========
Overcommitment rate: 0.7184466019417476 (222 / 309)
Undercommitment rate: 0.06472491909385113 (20 / 309)
========== /Users/andrew/Desktop/Code/_penn/redel-experiments/experiments/fanoutqa/cohere-hf/small-baseline ==========
Overcommitment rate: 1.0 (307 / 307)
Undercommitment rate: 0.0 (0 / 307)
========== /Users/and

In [10]:
df[(df["system"] == "full") | (df["system"] == "small-all")]

,model,benchmark,system,n,oc_count,oc_rate,uc_count,uc_rate
0,cohere-hf,fanoutqa,full,310,310,1.000000,0,0.000000
3,cohere-hf,fanoutqa,small-all,309,222,0.718447,20,0.064725
7,cohere-hf,travelplanner,full,180,83,0.461111,0,0.000000
10,cohere-hf,travelplanner,small-all,102,54,0.529412,1,0.009804
14,cohere-hf,webarena,full,213,212,0.995305,0,0.000000
16,glm,fanoutqa,full,290,148,0.510345,92,0.317241
21,glm,travelplanner,full,180,180,1.000000,0,0.000000
26,gpt-oss,fanoutqa,full,310,223,0.719355,18,0.058065
30,gpt-oss,fanoutqa,small-all,310,247,0.796774,3,0.009677
34,gpt-oss,travelplanner,full,180,180,1.000000,0,0.000000


Getting score conditional on over/undercommitted results.

In [6]:
# foqa
benchmark = "fanoutqa/dev/trial2"
system = "full"

with open(EXPERIMENTS / benchmark / system / "score.json") as f:
    fo_scores = json.load(f)

commitment = count_system(EXPERIMENTS / benchmark / system)
bad_ids = set(commitment.uc_ids) | set(commitment.uc_ids)

good_scores = [s for s in fo_scores["raw"] if s["question_id"] in bad_ids]

good_loose = sum(s["acc"] for s in good_scores) / len(good_scores)
good_gpt = sum(s["gpt"] for s in good_scores) / len(good_scores)

print("========== FOQA ==========")
print(f"Full Loose: {fo_scores['acc']['loose']}")
print(f"Full GPT: {fo_scores['gpt']}")
print(f"Filtered Loose: {good_loose}")
print(f"Filtered GPT: {good_gpt}")

# for system in ["full", "small-leaf"]:

FileNotFoundError: [Errno 2] No such file or directory: '/Users/andrew/Desktop/Code/kanpai/experiments/fanoutqa/dev/trial2/full/score.json'

In [7]:
len(bad_ids)

NameError: name 'bad_ids' is not defined

In [ ]:
bad_ids

In [ ]:
# foqa
benchmark = "fanoutqa/dev/trial2"
system = "full"

with open(EXPERIMENTS / benchmark / system / "score.json") as f:
    fo_scores = json.load(f)

fails = [s for s in fo_scores["raw"] if s["gpt"] == 0]
fail_ids = [s["question_id"] for s in fails]


# for system in ["full", "small-leaf"]:

In [ ]:
fails